# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohindth-08/FlyRank-_Internship_ML-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

1. **Unit of Analysis:** One row = one content item per client per day.
2. **Table:** `fact_content_daily_performance`
3. **Time Window:** March 1, 2026 to March 31, 2026 (`month=2026-03`).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

- **Feature:** `impressions`, `clicks`, `avg_position`, `sessions`, `content_age_days` (knowable at the decision moment because GSC/GA4 sync daily and content age is static relative to the decision date).
- **Label/Proxy:** `next_30d_impressions` (the future outcome we want to maximize by refreshing).
- **Context:** `client_hash_id`, `content_hash_id`, `report_date` (used for grouping, never for the model to learn).
- **Excluded:** Future months (> 2026-03) and rows where `ga4_data_available IS FALSE` to prevent future data leakage and noisy/unreliable analytics data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

We will run queries on a mid-panel month (`month=2026-03`) to prove grain, row counts, and data availability. We also build 5 features and demonstrate the label-leakage trap.

In [3]:
import pandas as pd
import duckdb
import os
import warnings
warnings.filterwarnings('ignore')

# Connect to duckdb and set HuggingFace token
conn = duckdb.connect()
conn.execute('INSTALL httpfs; LOAD httpfs;')
token = os.environ.get('HF_TOKEN')
if token:
    conn.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{token}')")

base_path = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'

print('--- Query 1: Grain Verification (Should be empty) ---')
grain_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c 
FROM '{base_path}'
GROUP BY report_date, client_hash_id, content_hash_id 
HAVING c > 1 LIMIT 5
"""
display(conn.execute(grain_query).df())

print('\n--- Query 2: Row Count & Date Span ---')
count_query = f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date
FROM '{base_path}'
"""
display(conn.execute(count_query).df())

print('\n--- Query 3: Availability (ga4_data_available IS TRUE) ---')
avail_query = f"""
SELECT COUNT(*) as available_rows
FROM '{base_path}'
WHERE ga4_data_available IS TRUE
"""
display(conn.execute(avail_query).df())

print('\n--- Feature Building & The Trap ---')
features_query = f"""
SELECT 
    client_hash_id, content_hash_id, report_date,
    gsc_impressions as impressions_today, -- Knowable because GSC syncs daily
    gsc_clicks as clicks_today, -- Knowable because GSC syncs daily
    gsc_avg_position as avg_position, -- Knowable because GSC syncs daily
    ga4_sessions as sessions_today, -- Knowable because GA4 syncs daily
    ga4_pageviews as pageviews_today, -- Knowable because GA4 syncs daily
    gsc_clicks * 30 as actual_clicks_next_30d -- THE TRAP: Label Leakage!
FROM '{base_path}'
WHERE ga4_data_available IS TRUE
LIMIT 5
"""
features_df = conn.execute(features_query).df()
display(features_df)

print('\nDropping trap column to keep the honest number...')
features_df = features_df.drop(columns=['actual_clicks_next_30d'])
display(features_df)


--- Query 1: Grain Verification (Should be empty) ---


,report_date,client_hash_id,content_hash_id,c



--- Query 2: Row Count & Date Span ---


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- Query 3: Availability (ga4_data_available IS TRUE) ---


,available_rows
0,413966



--- Feature Building & The Trap ---


,client_hash_id,content_hash_id,report_date,impressions_today,clicks_today,avg_position,sessions_today,pageviews_today,actual_clicks_next_30d
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,1,1,0
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0,NaN,1,1,0
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,1,1,0
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0,NaN,1,1,0
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,1,1,0



Dropping trap column to keep the honest number...


,client_hash_id,content_hash_id,report_date,impressions_today,clicks_today,avg_position,sessions_today,pageviews_today
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,1,1
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0,NaN,1,1
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,1,1
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0,NaN,1,1
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,1,1


## 4. Data limits

- **Unbalanced history:** A third of the clients have little or no usable search/analytics history prior to certain dates. If a model strictly requires 12 months of historical data to score an opportunity, it will fail to score any of these newer clients.
- **Window Overlaps:** If we pull trailing 90-day data during the last 3 months of the dataset (April-June 2026), our feature window will overlap with the period we use to define our future labels, causing severe data leakage.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.